# data-collection_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (128).ipynb`

| Field | Value |
|---|---|
| Section | `data_collection` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 4 |
| Detected functions | save_parquet_by_ticker |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import pandas_datareader as pdr
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import os

def save_parquet_by_ticker(input_file, output_directory):
    # Read the input Parquet file
    df = pd.read_parquet(input_file)
    
    # Ensure the output directory exists
    os.makedirs(output_directory, exist_ok=True)
    
    # Group the DataFrame by the 'Ticker' column
    grouped = df.groupby('Ticker')
    
    # Save each group to a separate Parquet file
    for ticker, group in grouped:
        # Construct a filename for the ticker
        ticker_safe = ticker.replace('.', '_')  # Replace '.' with '_' to make filename safe
        output_file = os.path.join(output_directory, f"{ticker_safe}.parquet")
        
        # Write the group to a Parquet file
        group.to_parquet(output_file, index=False)
        print(f"Saved {ticker} data to {output_file}")

# Example usage
input_file = 'SEP/SEP_2024.parquet'  # Path to the input Parquet file
output_directory = 'SEP/TickerFiles'  # Directory to save the separate Parquet files
save_parquet_by_ticker(input_file, output_directory)


In [ ]:
import os
import pandas as pd

# Define the base path and folders
base_path = ""  # Update with your directory path
folders = ["AUG/TickerFiles", "SEP/TickerFiles", "OCT/TickerFiles"]
output_folder = os.path.join(base_path, "Quarter")  # Ensure the path is absolute

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Collect all unique file names across the folders
all_files = set()
for folder in folders:
    folder_path = os.path.join(base_path, folder)
    if os.path.exists(folder_path):
        all_files.update(os.listdir(folder_path))
    else:
        print(f"Warning: Folder '{folder_path}' does not exist. Skipping.")

print("Starting the merging process...\n")

for file_name in all_files:
    if not file_name.endswith(".parquet"):  # Skip non-Parquet files
        print(f"Skipping non-Parquet file: {file_name}")
        continue

    output_file_path = os.path.join(output_folder, file_name)
    print(f"Merging Parquet file: {file_name}")

    merged_data = pd.DataFrame()  # Initialize an empty DataFrame for merging

    for folder in folders:
        input_file_path = os.path.join(base_path, folder, file_name)
        if os.path.exists(input_file_path):
            print(f"  - Reading from: {input_file_path}")
            try:
                # Read the Parquet file
                data = pd.read_parquet(input_file_path)
                merged_data = pd.concat([merged_data, data], ignore_index=True)
            except Exception as e:
                print(f"    Error reading {input_file_path}: {e}")
        else:
            print(f"  - File not found in {folder}, skipping.")

    # Write the merged DataFrame to a new Parquet file
    if not merged_data.empty:
        merged_data.to_parquet(output_file_path, index=False)
        print(f"Finished merging {file_name} into {output_file_path}\n")
    else:
        print(f"No data to merge for {file_name}. Skipping output.\n")

print("All files have been successfully merged!")
